# LLM News Research Tool
## Notebook 3: Ranking the articles

This notebook covers the third stage of the tool: deciding the order of the kept articles and which k articles move on to the summary step. It compares orderings with and without the relevance model, tests the best of them on a fresh week that is labelled blind and frozen before any model runs, and asks whether the model's help is worth its cost on the free plans.

Every table is computed from saved records and saved model answers. No model and no search is called.

In [1]:
import csv, json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
from src import ranking as rk
from src.config import EVAL_DIR
from src.fulltext_eval import _inputs

FULL_TEXT_HERE = len(_inputs("v3_dev_tata_motors_checked.csv")[2]) > 0
saved_threshold = json.loads((EVAL_DIR / "ranking_dev_record.json").read_text(encoding="utf-8"))["threshold"]
saved_report = json.loads((EVAL_DIR / "model_compare" / "report.json").read_text(encoding="utf-8"))
pd.set_option("display.max_colwidth", 70)

## 1. Orderings, selections and measures

| Ordering | Rule |
|---|---|
| O1 | where the company is named: the title, then the description, then only the body; ties keep NewsAPI's order |
| O2 | the model's label: `direct`, then `indirect`, then `not` |
| O3 | as O2, but an article that names the company in its title and is answered `not` goes with the `indirect` group |

| Selection | Rule |
|---|---|
| S1 | the first k of the ordering |
| S2 | the first k, skipping an article too similar to one already chosen |

An article without a model answer goes with the `indirect` group, so a failed answer never removes it. The measures, for the top k of each search day:

- **precision**: relevant articles among the k;
- **event recall**: relevant events with an article in the top k, out of the relevant events among the day's candidates;
- **direct-event recall**: the same for events with at least one `direct` article.

They measure the candidates each search returns, not all news published. A configuration replaces the baseline O1 + S1 only if it gains an event (at least one direct event, or two events), loses no direct event, keeps event recall and precision at least as high, answers within 180 seconds and falls back safely.

## 2. The similarity threshold for S2

S2 treats two articles as copies of one story when the share of content words they share reaches a threshold. The threshold matters because S2 exists to fill the k places with different events: a *false merge* (two different events treated as copies) drops an event from the selection, and a *missed merge* (two articles on the same event kept apart) spends a second place on an event already chosen.

The threshold is chosen on the development candidates. Every pair of candidates that has a hand-made event (the relevant ones) is checked once; an article found on more than one search day counts once. A pair is a false merge when S2 treats it as copies although the events differ, and a missed merge when the events are the same but S2 keeps the articles apart. **Grouping errors** are the two added together. They describe how well S2 recognises copies, not how well a configuration ranks. The tested value with the fewest errors is used.

In [2]:
from src.ranking_eval import DEV_RUNS, _load_run, choose_threshold, stage1_candidates
from src.config import DATA_DIR
from src.fulltext import article_id
from src.query import load_aliases
from src.summary_run import EVENTS_FILE, load_events

if FULL_TEXT_HERE:
    aliases = load_aliases(DATA_DIR / "company_aliases.json")
    full_texts = _inputs("v3_dev_tata_motors_checked.csv")[2]
    unique = {c["key"]: c for files in DEV_RUNS.values()
              for c in stage1_candidates(_load_run(files), full_texts, "Tata Motors", aliases)}
    events = {article_id(url): e for url, e in load_events(EVAL_DIR / EVENTS_FILE).items()}
    choice = choose_threshold(list(unique.values()), events)
    q0 = next(iter(choice["quality"].values()))
    print(f"{len(unique)} distinct candidates, {sum(k in events for k in unique)} with an event, {q0['pairs']} pairs.")
    display(pd.DataFrame({t: {"false merges": q["false_merges"], "missed merges": q["missed_merges"],
                              "grouping errors": choice["errors"][t]} for t, q in choice["quality"].items()}).T)
    assert {str(t): e for t, e in choice["errors"].items()} == saved_threshold["errors"]
    assert choice["threshold"] == saved_threshold["chosen"] == saved_report["threshold"]
    print(f"Chosen threshold: {choice['threshold']}. The errors equal the saved development record.")
else:
    print("Grouping errors (saved development record):", saved_threshold["errors"])
    print("Chosen threshold:", saved_threshold["chosen"])

55 distinct candidates, 41 with an event, 820 pairs.


,false merges,missed merges,grouping errors
0.3,0,27,27
0.4,0,33,33
0.5,0,34,34
0.6,0,34,34
0.7,0,34,34


Chosen threshold: 0.3. The errors equal the saved development record.


The chosen value, 0.3, sits at the edge of the tested range. An exploratory check on the same development candidates, made after the fresh test, adds values below 0.3. It does not change the frozen threshold.

In [3]:
if FULL_TEXT_HERE:
    lower = choose_threshold(list(unique.values()), events, thresholds=(0.15, 0.2, 0.25, 0.3))
    display(pd.DataFrame({t: {"false merges": q["false_merges"], "missed merges": q["missed_merges"],
                              "grouping errors": lower["errors"][t]} for t, q in lower["quality"].items()}).T)
else:
    print("The development full texts (data/fulltext/, outside Git) are not on this computer, so this check "
          "cannot be recomputed here; the text below states its result.")

,false merges,missed merges,grouping errors
0.15,4,10,14
0.20,1,18,19
0.25,0,22,22
0.30,0,27,27


Below 0.3 there are fewer grouping errors. 0.15 has the fewest (14), but 4 of them are false merges, and each false merge can drop an event. 0.25 has fewer errors than 0.3 and still no false merges. Which value is better depends on how a false merge is weighed against a missed merge, and the rule weighs them equally. The frozen 0.3 stays: the fresh test is scored once, and S2 is not part of the default. A lower threshold needs its own development test and a new fresh week.

## 3. Comparing models on the free plans

Each configuration answers the same development candidates with the same prompt. One row per configuration and attempt: both Gemini attempts and both Qwen settings stay visible. *Best ranking* is the model-based ranking that passes the rule on the most search days for that configuration, and *days passing* is its count. *Coverage* is the share of articles the model answers; an article without an answer falls back. A run with coverage below 90% is a reliability result only, not valid for quality.

In [4]:
def prompt_version(commit):
    try:
        text = subprocess.run(["git", "show", f"{commit}:src/relevance_check.py"], capture_output=True,
                              text=True, check=True).stdout
        return next(line.split("=")[1].strip().strip('"') for line in text.splitlines()
                    if line.startswith("PROMPT_VERSION"))
    except Exception:
        return "unknown"

rows = []
for path in sorted((EVAL_DIR / "model_compare").glob("run_*.json")):
    run = json.loads(path.read_text(encoding="utf-8"))
    name = run["configuration"] + ("" if run.get("attempt", 1) == 1 else f" (attempt {run['attempt']})")
    s, m = saved_report["summaries"][name], saved_report["details"][name]["measures_pooled"]
    settings = run["settings"]["generation"]
    rows.append({"configuration": name, "provider": run["settings"]["provider"], "model": run["settings"]["model"],
                 "reasoning": settings.get("reasoning_effort") or settings.get("thinking_level"),
                 "prompt": prompt_version(run["commit"]), "coverage": round(m["coverage"], 3),
                 "failures": m["fallbacks"] or "none", "slowest search day (s)": s["seconds"],
                 "best ranking": s["ranking"], "days passing (of 3)": s["runs_passed"], "valid for quality": m["coverage"] >= 0.9})
pd.DataFrame(rows).set_index("configuration")

,provider,model,reasoning,prompt,coverage,failures,slowest search day (s),best ranking,days passing (of 3),valid for quality
configuration,,,,,,,,,,
gemini-3.8-flash,gemini,gemini-3.8-flash,medium,3,0.091,"{'server': 37, 'daily_limit': 13}",187.7,O3+S1,0,False
gemini-3.8-flash (attempt 2),gemini,gemini-3.8-flash,medium,3,0.309,{'server': 38},326.7,O3+S1,0,False
gpt-oss-120b,groq,openai/gpt-oss-120b,medium,3,1.000,none,274.2,O2+S1,1,True
gpt-oss-20b,groq,openai/gpt-oss-20b,medium,3,1.000,none,273.5,O2+S1,1,True
qwen3.8-27b-low,groq,qwen/qwen3.8-27b,low,3,0.927,{'too_large': 4},900.9,O3+S1,0,True
qwen3.8-27b,groq,qwen/qwen3.8-27b,medium,3,0.909,{'too_large': 5},907.5,O3+S1,0,True


In these development runs only the two gpt-oss models answer every article. Gemini's free tier fails on server overload and its daily limit in both attempts, and Qwen's requests are sometimes too large for the free per-minute limit. Every model configuration also takes longer than 180 seconds on its slowest search day. No configuration passes the rule on two of the three days, so the development choice is the baseline, O1 + S1.

## 4. Original and checked labels

*Model F1* compares the model's relevant / not relevant answers with the human labels. *Events at 10* counts the relevant events in the top 10 of the configuration's best ranking, added over the three days.

The comparison is recomputed twice: with the original labels (guide version 3), where it must equal the saved report, and with the checked labels from the blind re-check (section 6). The answers, prompts, orderings and threshold stay the same.

In [5]:
from src.model_compare import build_report

if FULL_TEXT_HERE:
    original = json.loads(json.dumps(build_report("Tata Motors", "v3_dev_tata_motors.csv"), default=str))
    assert original == saved_report, "the recomputed comparison differs from the saved report"
    checked = json.loads(json.dumps(build_report("Tata Motors", "v3_dev_tata_motors_checked.csv"), default=str))
    rows = {}
    for name in original["summaries"]:
        o, c = original["summaries"][name], checked["summaries"][name]
        rows[name] = {"days passing (original)": o["runs_passed"], "days passing (checked)": c["runs_passed"],
                      "events at 10 (original)": o["events_k10"], "events at 10 (checked)": c["events_k10"],
                      "model F1 (original)": round(o["f1"], 3), "model F1 (checked)": round(c["f1"], 3)}
    display(pd.DataFrame(rows).T.astype({k: int for k in next(iter(rows.values())) if "F1" not in k}))
    print("The recomputed comparison equals the saved report.")
    print(f"Development choice: original labels {original['choice']}, checked labels {checked['choice']}.")
    changed = [n for n in rows if original["summaries"][n]["runs_passed"] != checked["summaries"][n]["runs_passed"]]
    print("Configurations whose days passing change:", ", ".join(changed) or "none")
else:
    print("The full texts are not on this computer; the saved report's choice:", saved_report["choice"])

,days passing (original),days passing (checked),events at 10 (original),events at 10 (checked),model F1 (original),model F1 (checked)
gemini-3.8-flash,0,0,18,18,0.136,0.136
gemini-3.8-flash (attempt 2),0,0,17,17,0.453,0.453
gpt-oss-120b,1,1,22,22,0.902,0.902
gpt-oss-20b,1,1,21,21,0.927,0.927
qwen3.8-27b-low,0,0,18,18,0.937,0.937
qwen3.8-27b,0,0,18,18,0.962,0.962


The recomputed comparison equals the saved report.
Development choice: original labels O1, checked labels O1.
Configurations whose days passing change: none


The one corrected label changes no measure in this table and no decision. The fresh test (section 5) is scored once and is not rescored.

## 5. The fresh test

The fresh test uses a new week (24 to 30 October 2026). Its articles are labelled blind and grouped into events before any model answers them, and the protocol (the configurations, the rule, the model and its settings, the fingerprints of the labels and events) is committed and tagged in Git first. The result is scored once.

In [6]:
from src.fresh_eval import protocol_problems

run_dir = EVAL_DIR / "fresh_2026-10-24_30"
protocol = json.loads((run_dir / "fresh_protocol.json").read_text(encoding="utf-8"))
fresh = json.loads((run_dir / "fresh_ranking_record.json").read_text(encoding="utf-8"))
problems = protocol_problems(protocol, run_dir / "labels.csv", run_dir / "events.csv")
assert not problems, problems
print("The labels and events match the frozen protocol.")
print(f"Candidates: {fresh['report']['candidates']}, relevant: {fresh['report']['relevant_candidates']}. "
      f"First model pass: {fresh['first_pass_runtime']['seconds']} s, coverage {fresh['first_pass_runtime']['coverage']}.")
rows = {}
for name, verdict in fresh["report"]["acceptance"].items():
    failed = [c for c, ok in verdict["conditions"].items() if not ok]
    rows[name] = {"passes": verdict["passed"], "failed conditions": ", ".join(failed) or "none",
                  "events lost": ", ".join(verdict["events"]["lost"]) or "none"}
display(pd.DataFrame(rows).T)
print("Default after the fresh test:", fresh["default"])

The labels and events match the frozen protocol.
Candidates: 36, relevant: 28. First model pass: 242.1 s, coverage 1.0.


,passes,failed conditions,events lost
O1+S2,False,gain,none
O3+S1,False,"gain, runtime",none
O3+S2,False,"gain, runtime",none
O2+S1,False,"gain, no_direct_loss, event_recall_not_lower, runtime",tata_motors_q2_fy27_results
O2+S2,False,"gain, no_direct_loss, event_recall_not_lower, runtime",tata_motors_q2_fy27_results


Default after the fresh test: O1+S1


## 6. The blind re-check of the labels

The same labeller labels a sample of development articles a second time, without seeing the first labels. Part A is 40 articles drawn from the development articles that are still unexposed (never shown with a label or in a fact sheet); agreement and Cohen's kappa come from Part A only, so they measure consistency on that subset, not on all development articles. Part B covers articles where the model and the first label disagree; those articles are exposed, so they are reviewed separately and stay out of the kappa.

The re-check pool is the development articles labelled on their own text: from the full text, or from the short text when the publisher blocks the page. The other labelled articles are sale listings, labelled on their short text, and copies of a fetched article.

In [7]:
from collections import Counter
from src.recheck import recheck_pool

with open(EVAL_DIR / "v3_dev_tata_motors.csv", encoding="utf-8-sig", newline="") as handle:
    label_rows = list(csv.DictReader(handle))
display(pd.Series(Counter(row["basis"] for row in label_rows), name="labelled articles").to_frame())
print(f"Labelled articles: {len(label_rows)}; in the re-check pool: {len(recheck_pool(label_rows))}.")

,labelled articles
full text,71
short text (page blocked by robots.txt),3
short text (copy of a fetched article),5
"short text (sale listing, not fetched)",25


Labelled articles: 104; in the re-check pool: 74.


In [8]:
score = json.loads((EVAL_DIR / "recheck_v3_score.json").read_text(encoding="utf-8"))
result = json.loads((EVAL_DIR / "recheck_v3_result.json").read_text(encoding="utf-8"))
a = score["part_a"]
record = json.loads((EVAL_DIR / "recheck_v3_record.json").read_text(encoding="utf-8"))
unexposed = record["pool_size"] - len(record["exposed"])
print(f"Development articles: {record['pool_size']}, exposed before the re-check: {len(record['exposed'])}, "
      f"unexposed: {unexposed}.")
print(f"Part A: {a['articles']} of the {unexposed} unexposed articles, agreement {a['agreement']:.3f}, Cohen's kappa {a['kappa']:.3f}.")
display(pd.DataFrame(a["per_label"]).T.astype({"articles": int, "same": int}).round(3)
        .rename(columns={"same": "same label"}))
print("Limit recorded with the score:", a["limits"])
print(f"Part B: {len(score['part_b'])} articles, reviewed separately.")
print(f"Adjudication: {result['decisions']} decisions, first label kept {result['first_label_kept']}, "
      f"re-check label taken {result['recheck_label_taken']}; corrections written: {result['corrections']}.")

Development articles: 74, exposed before the re-check: 26, unexposed: 48.
Part A: 40 of the 48 unexposed articles, agreement 0.825, Cohen's kappa 0.723.


,articles,same label,agreement
direct,12,12,1.000
indirect,9,5,0.556
not,19,16,0.842


Limit recorded with the score: Part A covers the unexposed development subset only, not all 74 articles. The fact-check sample holds the long relevant articles, so long relevant articles are underrepresented in Part A.
Part B: 8 articles, reviewed separately.
Adjudication: 10 decisions, first label kept 9, re-check label taken 1; corrections written: 1.


Agreement is complete on `direct` and lowest on `indirect`, the class the guide finds hardest to define. The checked labels, with the one correction, are the labels used from this point on.

## 7. Error analysis: O2 and O3

O2 orders only by the model's label, so a wrong `not` pushes an important article down. On the fresh test this happens: O2 loses a direct event that O1 keeps (section 5). O3 is the attempted improvement: it protects articles that name the company in their title from being pushed down by a `not`. On the fresh test O3 loses no event. But O3, like every configuration that uses the model, still fails the 180-second rule: the first model pass takes longer than that on the free plan. The bottleneck is the time to get the model's answers, not the title protection, which is only a sorting rule on answers already given.

In [9]:
times = {name: fresh["report"]["acceptance"][name]["conditions"].get("runtime") for name in fresh["report"]["acceptance"]}
print("Runtime condition for each model configuration:", {n: ok for n, ok in times.items() if ok is not None})
print("Direct events lost:", {n: v["events"]["direct_lost"] for n, v in fresh["report"]["acceptance"].items()})

Runtime condition for each model configuration: {'O3+S1': False, 'O3+S2': False, 'O2+S1': False, 'O2+S2': False}
Direct events lost: {'O1+S2': [], 'O3+S1': [], 'O3+S2': [], 'O2+S1': ['tata_motors_q2_fy27_results'], 'O2+S2': ['tata_motors_q2_fy27_results']}


## 8. A second development cycle: articles without the company in the headline

A later test on a second company, Reliance Industries, finds a weakness in O1. An article whose headline and description do not name the company ranks below every article whose headline does, even when it is the most important news of the week. There, it costs one direct event. The fix is built and tested on Tata Motors development data only, so the Reliance labels do not shape the pipeline.

**Four variants** change O1 in one way each. Everything else stays the same, including S1 and the k values.

| Variant | Rule |
|---|---|
| O1R1 | the first candidate in retrieval order goes to the top, then O1 |
| O1R2 | the first two candidates in retrieval order go to the top, then O1 |
| O1M3 | a candidate whose headline does not name the company, but whose text names it at least 3 times, joins the headline group |
| O1M5 | the same with at least 5 names |

Names are counted as whole words, longest name first and without overlaps; a name that needs a context word counts only with it.

**The rule, fixed and tagged before any result** (`ranking_variants_plan.json`, tag `freeze-ranking-variants-plan`): a variant passes only if, on every development day at k = 10, it loses no direct event against O1 + S1 and has event recall and precision that are not lower (a tie is not lower). The passing variants are then ranked by the direct events they cover over the three days, then all events, then pooled precision, then simplicity. If none passes, O1 + S1 stays.


In [10]:
variants = json.loads((EVAL_DIR / "ranking_variants_dev_record.json").read_text(encoding="utf-8"))
rows = []
for day, result in variants["days"].items():
    for name, measures in result["measures"].items():
        m = measures["10"]
        check = result["checks"].get(name)
        rows.append({"day": day, "ordering": "O1 + S1" if name == "O1" else name,
                     "precision": round(m["precision"], 2),
                     "events": f"{len(m['events_covered'])} of {m['relevant_events']}",
                     "direct events": f"{len(m['direct_events_covered'])} of {m['direct_events']}",
                     "lost direct events": ", ".join(check["events"]["direct_lost"]) if check else "",
                     "passes the day": "" if check is None else ("yes" if check["passed"] else "no")})
display(pd.DataFrame(rows).set_index(["day", "ordering"]))

pooled = pd.DataFrame(variants["choice"]["pooled"]).T.rename(index={"O1": "O1 + S1"})
pooled = pooled.astype({"direct_events_covered": int, "events_covered": int})
pooled["pooled_precision"] = pooled["pooled_precision"].astype(float).round(2)
display(pooled)
print("Passing on every day:", variants["choice"]["passing"], "| chosen:", variants["choice"]["chosen"])

week1 = json.loads((EVAL_DIR / "ranking_variants_week1_check.json").read_text(encoding="utf-8"))
for key, place in week1["articles"].items():
    print(f"Reliance article {key}: retrieval place {place['retrieval']}, O1 place {place['O1']}, "
          f"O1M5 place {place['O1M5']} of {week1['candidates']}; in the top 10: {place['in_top_10']}")


precision   events direct events  \
day        ordering                                     
2026-10-14 O1 + S1         1.0   5 of 9        4 of 4   
           O1R1            0.9   5 of 9        4 of 4   
           O1R2            0.9   5 of 9        4 of 4   
           O1M3            1.0   6 of 9        4 of 4   
           O1M5            1.0   5 of 9        4 of 4   
2026-10-23 O1 + S1         0.8   6 of 9        4 of 4   
           O1R1            0.8   6 of 9        4 of 4   
           O1R2            0.8   6 of 9        4 of 4   
           O1M3            0.9   7 of 9        4 of 4   
           O1M5            0.8   6 of 9        4 of 4   
2026-10-24 O1 + S1         0.7  7 of 13        5 of 6   
           O1R1            0.7  7 of 13        5 of 6   
           O1R2            0.7  7 of 13        5 of 6   
           O1M3            0.8  8 of 13        5 of 6   
           O1M5            0.8  8 of 13        6 of 6   

                            lost direct events passes the day  
day        ordering                                            
2026-10-14 O1 + S1                                             
           O1R1                                            no  
           O1R2                                            no  
           O1M3                                           yes  
           O1M5                                           yes  
2026-10-23 O1 + S1                                             
           O1R1                                           yes  
           O1R2                                           yes  
           O1M3                                           yes  
           O1M5                                           yes  
2026-10-24 O1 + S1                                             
           O1R1                                           yes  
           O1R2                                           yes  
           O1M3      range_rover_electric_test             no  
           O1M5                                           yes

,direct_events_covered,events_covered,pooled_precision
O1 + S1,13,18,0.83
O1R1,13,18,0.80
O1R2,13,18,0.80
O1M3,13,21,0.90
O1M5,14,19,0.87


Passing on every day: ['O1M5'] | chosen: O1M5
Reliance article A6327d7dcd9: retrieval place 1, O1 place 47, O1M5 place 50 of 53; in the top 10: False


Only **O1M5** passes on all three days. O1R1 and O1R2 put a not-relevant article into the top 10 on 14 October, so their precision drops from 1.00 to 0.90. O1M3 loses a direct event on 24 October ("range_rover_electric_test") while gaining another, and the rule allows no lost direct event. O1M5 matches O1 + S1 on 14 and 23 October. On 24 October it gains a direct event ("jlr_rebrand_narrative"), and its precision rises from 0.70 to 0.80. Over the three days it covers 14 direct events against 13, and 19 events against 18, with pooled precision 0.87 against 0.83. **O1M5 is chosen.**

**The Reliance article.** A check on the Reliance week, fixed in advance as information only, shows that O1M5 does not rescue the article behind the lost event. It names the group mostly through short forms such as "Jio", which are not names in the alias file, so it does not reach five names. It is first in retrieval order, 47th of 53 under O1 and 50th under O1M5, and outside the top 10 under both. The aliases are not changed to fix this, because the Reliance labels must not shape the pipeline. Section 9 tests it on two fresh weeks.

O1M5 becomes the default ranking at the final freeze of this cycle. It is chosen on three Tata Motors development days, so the gain is one direct event on one day, which is small evidence.


## 9. The final default on two fresh weeks

The second development cycle replaces O1 with O1M5, so the final default is **O1M5 + S1**. No model-based ordering earns its place on the free plans, so the default orders without a model call; the model review stays available as an optional, clearly labelled feature. Two fresh weeks, one for Tata Motors and one for Reliance Industries, test O1M5 + S1 at k = 10 against the retrieval order and against O1 + S1.

In [11]:
week2_folders = {"Tata Motors": "week2_tata_2026-10-31_11-06", "Reliance Industries": "week2_reliance_2026-11-02_11-08"}
rows, checks = [], []
for company, folder in week2_folders.items():
    record = json.loads((EVAL_DIR / folder / "week2_ranking.json").read_text(encoding="utf-8"))
    lists = {"retrieval order": record["primary"]["absolute"]["retrieval_order"]["10"],
             "O1 + S1": record["second"]["absolute"]["O1+S1"]["10"],
             "O1M5 + S1": record["primary"]["absolute"]["O1M5+S1"]["10"]}
    covered = {"retrieval order": record["primary"]["events_covered"]["retrieval_order"]["10"],
               "O1M5 + S1": record["primary"]["events_covered"]["O1M5+S1"]["10"],
               "O1 + S1": record["second"]["events_covered"]["O1+S1"]["10"]}
    for name, m in lists.items():
        rows.append({"week": company, "list": name, "precision": round(m["precision"], 2),
                     "events": len(covered[name]), "event recall": round(m["event_recall"], 3),
                     "direct event recall": round(m["direct_event_recall"], 3)})
    for label, part in (("against the retrieval order", record["primary"]),
                        ("against O1 + S1", record["second"]),
                        ("O1 + S1 against the retrieval order", record["old_default_against_retrieval"])):
        checks.append({"week": company, "comparison": label, **part["conditions"],
                       "gained": ", ".join(part["events"]["gained"]), "lost": ", ".join(part["events"]["lost"])})
display(pd.DataFrame(rows).set_index(["week", "list"]))
display(pd.DataFrame(checks).set_index(["week", "comparison"]))

precision  events  event recall  \
week                list                                               
Tata Motors         retrieval order        1.0       7         0.389   
                    O1 + S1                1.0       5         0.278   
                    O1M5 + S1              1.0       6         0.333   
Reliance Industries retrieval order        0.8       4         0.222   
                    O1 + S1                0.8       3         0.167   
                    O1M5 + S1              0.8       4         0.222   

                                     direct event recall  
week                list                                  
Tata Motors         retrieval order                0.500  
                    O1 + S1                        0.500  
                    O1M5 + S1                      0.600  
Reliance Industries retrieval order                0.333  
                    O1 + S1                        0.250  
                    O1M5 + S1                      0.250

no_direct_event_lost  \
week                comparison                                                  
Tata Motors         against the retrieval order                          True   
                    against O1 + S1                                      True   
                    O1 + S1 against the retrieval order                 False   
Reliance Industries against the retrieval order                         False   
                    against O1 + S1                                      True   
                    O1 + S1 against the retrieval order                 False   

                                                         event_recall_not_lower  \
week                comparison                                                    
Tata Motors         against the retrieval order                           False   
                    against O1 + S1                                        True   
                    O1 + S1 against the retrieval order                   False   
Reliance Industries against the retrieval order                            True   
                    against O1 + S1                                        True   
                    O1 + S1 against the retrieval order                   False   

                                                         precision_not_lower  \
week                comparison                                                 
Tata Motors         against the retrieval order                         True   
                    against O1 + S1                                     True   
                    O1 + S1 against the retrieval order                 True   
Reliance Industries against the retrieval order                         True   
                    against O1 + S1                                     True   
                    O1 + S1 against the retrieval order                 True   

                                                                                      gained  \
week                comparison                                                                 
Tata Motors         against the retrieval order                tata_motors_share_price_surge   
                    against O1 + S1                      chandrasekaran_meloni_iveco_meeting   
                    O1 + S1 against the retrieval order        tata_motors_share_price_surge   
Reliance Industries against the retrieval order                           ambani_wealth_jump   
                    against O1 + S1                                       ambani_wealth_jump   
                    O1 + S1 against the retrieval order                                        

                                                                                                                          lost  
week                comparison                                                                                                  
Tata Motors         against the retrieval order                 dhanteras_festive_auto_deliveries, october_vahan_registrations  
                    against O1 + S1                                                                                             
                    O1 + S1 against the retrieval order  chandrasekaran_meloni_iveco_meeting, dhanteras_festive_auto_delive...  
Reliance Industries against the retrieval order                                           jio_comcast_charter_technology_talks  
                    against O1 + S1                                                                                             
                    O1 + S1 against the retrieval order                                   jio_comcast_charter_technology_talks

**Against O1 + S1**, O1M5 + S1 meets all three conditions in both weeks: it adds one event each week and loses none. The change does what it is chosen for on fresh data.

**Against the retrieval order**, it passes in neither week, for two different reasons. In the Tata Motors week, one large story fills four of the top ten places, because S1 does not group articles about the same news; event recall falls from 7 to 6 of 18. In the Reliance Industries week, a direct event is lost because its headline names the company only as "Jio", a short form that is not in the alias file. This is the same name-matching problem identified in the Reliance week 1 test (section 8). O1 + S1 also fails against the retrieval order in all three fresh tests that make this comparison, so the plain retrieval order is a strong baseline on fresh news.

Two separate experiments remain for a later cycle: grouping similar articles (S2), and counting a short name such as "Jio" when no other Jio company is named, with a check for false matches with Jio Financial Services. Neither is made the default here, because the fresh weeks are test data, and test data does not shape the pipeline.

## 10. Limitations

- The evidence comes from three Tata Motors development days, two fresh weeks, and the free plans of two providers.
- The measures cover the candidates the searches return, not all news published.